# Chapter 15: Discrete Bayes Filters

<a href="../lite/lab/index.html?path=ch15_discrete_bayes.ipynb" target="_blank" style="display:inline-block;padding:8px 18px;background:#1976d2;color:white;border-radius:5px;text-decoration:none;font-weight:bold;font-size:0.95em;">&#9654; Open in JupyterLite: run and edit this notebook</a>

*Runs entirely in your browser, no installation required.*

**How to use:** Edit the parameter values in each cell and re-run it to explore.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d
from scipy.signal import fftconvolve

%matplotlib inline
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

Forget calculus. Forget matrices. The simplest filter is just a list of numbers, one probability for each possible position. Move: shift and blur the list. Sense: multiply and renormalize. That is the entire algorithm. And it works.

The **discrete Bayes filter** (also called the **histogram filter**) is the most intuitive way to understand probabilistic state estimation. Every idea in later chapters (Kalman filters, particle filters) is a variation on this same predict/update loop. If you understand this chapter, the rest will follow naturally.

## Opening Demo: The Hallway Problem

A robot lives in a 1D hallway with 20 cells. Three of those cells have doors (at positions 4, 9, and 16). The robot has a sensor that can detect whether it is next to a door, but the sensor is noisy. The robot starts with **no idea** where it is (uniform belief). Watch what happens as it senses and moves.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_cells       = 20          # hallway length                (try 10, 20, 40)
door_cells    = [4, 9, 16]  # where the doors are           (try different combos)
p_hit         = 0.8         # P(sensor=door | at door)      (try 0.6, 0.8, 0.95)
p_miss        = 0.2         # P(sensor=door | no door)      (try 0.05, 0.2, 0.4)
move_distance = 1           # cells moved per step           (try 1, 2)
move_noise    = 1           # kernel half-width for blur     (try 0, 1, 2)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

# True robot position starts at cell 4 (on a door)
true_pos = 4

# Sensor readings the robot will get (simulated)
# Step 0: sense (at door 4), Step 1: move+sense, Step 2: move+sense, ...
scenario = [
    ('sense', True),          # at door 4: sees door
    ('move',  move_distance),
    ('sense', False),         # at cell 5: no door
    ('move',  move_distance),
    ('sense', False),         # at cell 6: no door
    ('move',  move_distance * 3),
    ('sense', True),          # at cell 9: sees door!
]

def make_motion_kernel(shift, noise_hw):
    """Create a motion convolution kernel: peak at 'shift', blurred by noise."""
    width = 2 * noise_hw + 1
    kernel = np.zeros(n_cells)
    for d in range(-noise_hw, noise_hw + 1):
        idx = (shift + d) % n_cells
        kernel[idx] = 1.0
    kernel /= kernel.sum()
    return kernel

def predict(belief, shift, noise_hw):
    """Motion update: shift and blur."""
    kernel = make_motion_kernel(shift, noise_hw)
    result = np.real(np.fft.ifft(np.fft.fft(belief) * np.fft.fft(kernel)))
    result = np.clip(result, 0, None)
    return result / result.sum()

def update(belief, measurement_is_door):
    """Measurement update: multiply by likelihood, normalize."""
    likelihood = np.array([
        p_hit if (i in door_cells) == measurement_is_door else (1 - p_hit)
        for i in range(n_cells)
    ])
    posterior = belief * likelihood
    return posterior / posterior.sum()

# Run the scenario and collect snapshots
belief = np.ones(n_cells) / n_cells  # uniform prior
snapshots = [('Initial (uniform)', belief.copy(), true_pos)]

pos = true_pos
for action, value in scenario:
    if action == 'sense':
        belief = update(belief, value)
        label = f'Sense {"door" if value else "no door"} (pos={pos})'
        snapshots.append((label, belief.copy(), pos))
    elif action == 'move':
        belief = predict(belief, value, move_noise)
        pos = (pos + value) % n_cells
        label = f'Move +{value} (now at {pos})'
        snapshots.append((label, belief.copy(), pos))

# Plot all snapshots
n_snaps = len(snapshots)
fig, axes = plt.subplots(n_snaps, 1, figsize=(12, 2.2 * n_snaps), sharex=True)
cells = np.arange(n_cells)

for ax, (label, bel, tpos) in zip(axes, snapshots):
    colors = ['orange' if i in door_cells else 'steelblue' for i in cells]
    ax.bar(cells, bel, color=colors, edgecolor='white', linewidth=0.5)
    ax.axvline(tpos, color='tomato', linewidth=2.5, linestyle='--', alpha=0.8, label=f'True pos = {tpos}')
    ax.set_ylabel('P')
    ax.set_title(label, fontsize=10, loc='left')
    ax.set_ylim(0, max(bel) * 1.3 + 0.01)
    ax.legend(fontsize=8, loc='upper right')

axes[-1].set_xlabel('Cell index')
axes[-1].set_xticks(cells)
fig.suptitle('Discrete Bayes Filter: Hallway Demo\n(orange bars = door cells)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('After sensing door, all three door positions light up.')
print('After moving and sensing "no door", non-door neighbors stay high.')
print('After a longer move and sensing door again, only one peak survives.')

## 15.1 Grid Representation

The core idea is simple: **discretize** the continuous state space into a finite grid of cells. Each cell holds a probability value, and the full array represents the robot's **belief** about where it is.

For a 1D world of length $L$ divided into $n$ cells:

$$\text{bel}(x) = [p_0, p_1, \dots, p_{n-1}], \quad \sum_{i=0}^{n-1} p_i = 1$$

Key properties:
- **Uniform prior** means $p_i = 1/n$ for all $i$ (total ignorance)
- A **peaked** distribution means the robot is confident about its position
- The distribution can be **multimodal** (multiple peaks), representing ambiguity

This representation makes no assumptions about the shape of the distribution, unlike the Gaussian assumption in Kalman filters. It can represent any distribution, including multimodal ones. The tradeoff: memory and computation grow with the number of cells.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_cells_demo  = 30     # number of grid cells        (try 10, 30, 100)
prior_type    = 'uniform'  # 'uniform', 'peaked', 'bimodal'
peak_center   = 15     # center of peak (for 'peaked')  (try 5, 15, 25)
peak_width    = 2.0    # std of peak (for 'peaked')     (try 0.5, 2.0, 5.0)
# ─────────────────────────────────────────────────────────────────────────────

cells = np.arange(n_cells_demo)

if prior_type == 'uniform':
    belief = np.ones(n_cells_demo) / n_cells_demo
elif prior_type == 'peaked':
    belief = np.exp(-0.5 * ((cells - peak_center) / peak_width) ** 2)
    belief /= belief.sum()
elif prior_type == 'bimodal':
    g1 = np.exp(-0.5 * ((cells - n_cells_demo * 0.3) / peak_width) ** 2)
    g2 = np.exp(-0.5 * ((cells - n_cells_demo * 0.7) / peak_width) ** 2)
    belief = g1 + g2
    belief /= belief.sum()
else:
    belief = np.ones(n_cells_demo) / n_cells_demo

fig, ax = plt.subplots(figsize=(11, 3))
ax.bar(cells, belief, color='steelblue', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Cell index')
ax.set_ylabel('Probability')
ax.set_title(f'Grid Belief Representation ({prior_type} prior, {n_cells_demo} cells)')
ax.set_ylim(0, belief.max() * 1.3 + 0.005)
plt.tight_layout()
plt.show()

print(f'Sum of probabilities: {belief.sum():.6f} (must be 1.0)')
print(f'Max probability cell: {np.argmax(belief)} with P = {belief.max():.4f}')
print(f'Entropy: {-np.sum(belief * np.log(belief + 1e-15)):.3f} bits')

## 15.2 Motion Update (Prediction)

When the robot moves, we must update the belief to reflect the motion. This is the **prediction step**. If the robot commands "move right by $d$ cells," we shift the entire probability distribution right by $d$.

But motion is never perfect. The robot might overshoot or undershoot. We model this uncertainty by **convolving** the shifted distribution with a small kernel:

$$\overline{\text{bel}}(x_i) = \sum_j p(x_i \mid x_j, u) \, \text{bel}(x_j)$$

In practice, this is a **shift + blur** operation:
1. **Shift** the distribution by the commanded distance
2. **Convolve** with a small kernel to spread probability (modeling motion noise)

The key insight: **prediction always increases uncertainty.** The distribution gets wider (more spread out) after every motion step. Information is lost, never gained, during prediction.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_cells_motion  = 40        # grid size                    (try 20, 40, 80)
start_cell      = 10        # initial peak position        (try 5, 10, 20)
start_sigma     = 1.5       # initial peak width           (try 0.5, 1.5, 3.0)
move_amount     = 8         # how far to move (cells)      (try 3, 8, 15)
motion_kernel   = [0.05, 0.1, 0.2, 0.3, 0.2, 0.1, 0.05]  # motion noise kernel
n_predict_steps = 5         # how many predictions to show (try 1, 3, 5, 10)
# ─────────────────────────────────────────────────────────────────────────────

cells = np.arange(n_cells_motion)

# Start with a peaked belief
belief = np.exp(-0.5 * ((cells - start_cell) / start_sigma) ** 2)
belief /= belief.sum()

# Normalize the kernel
kernel_arr = np.array(motion_kernel)
kernel_arr /= kernel_arr.sum()

def motion_update(bel, shift, kernel):
    """Shift by 'shift' cells, then convolve with kernel."""
    shifted = np.roll(bel, shift)
    # Pad for convolution, then trim
    padded = np.pad(shifted, len(kernel), mode='wrap')
    convolved = np.convolve(padded, kernel, mode='same')
    result = convolved[len(kernel):-len(kernel)]
    result = np.clip(result, 0, None)
    return result / result.sum()

# Collect snapshots after repeated prediction
snapshots = [('Before motion', belief.copy())]
bel = belief.copy()
for step in range(1, n_predict_steps + 1):
    bel = motion_update(bel, move_amount, kernel_arr)
    snapshots.append((f'After {step} motion step(s)', bel.copy()))

fig, axes = plt.subplots(len(snapshots), 1, figsize=(12, 2.0 * len(snapshots)), sharex=True)
for ax, (label, bel_snap) in zip(axes, snapshots):
    ax.bar(cells, bel_snap, color='steelblue', edgecolor='white', linewidth=0.5)
    ax.set_ylabel('P')
    ax.set_title(label, fontsize=10, loc='left')
    ax.set_ylim(0, snapshots[0][1].max() * 1.2)

axes[-1].set_xlabel('Cell index')
fig.suptitle('Motion Update: Each step shifts and blurs the distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Notice how the peak shifts right and gets broader with each step.')
print('Prediction never sharpens the belief; it always adds uncertainty.')

## 15.3 Measurement Update

When the robot takes a sensor reading, we incorporate that information using **Bayes' rule**. This is the **update step** (also called the **correction step**).

$$\text{bel}(x_i) = \eta \, p(z \mid x_i) \, \overline{\text{bel}}(x_i)$$

where:
- $p(z \mid x_i)$ is the **likelihood** of getting measurement $z$ if the robot is at cell $i$
- $\overline{\text{bel}}(x_i)$ is the predicted belief (from the motion step)
- $\eta$ is a normalizing constant so probabilities sum to 1

The operation is simple:
1. **Multiply** each cell's probability by how likely the measurement would be from that cell
2. **Normalize** so everything sums to 1

The key insight: **measurement always decreases uncertainty.** Cells that are inconsistent with the measurement get their probability reduced. The distribution becomes sharper.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_cells_meas    = 30        # grid size                        (try 20, 30, 50)
landmark_cells  = [8, 20]   # cells with landmarks             (try [8, 20], [5, 15, 25])
sensor_accuracy = 0.85      # P(detect | at landmark)          (try 0.6, 0.85, 0.99)
false_positive  = 0.1       # P(detect | not at landmark)      (try 0.01, 0.1, 0.3)
# ─────────────────────────────────────────────────────────────────────────────

cells = np.arange(n_cells_meas)

# Start with a broad belief (slightly noisy uniform)
prior = np.ones(n_cells_meas) / n_cells_meas

# Likelihood function: robot senses "landmark detected"
likelihood = np.array([
    sensor_accuracy if i in landmark_cells else false_positive
    for i in range(n_cells_meas)
])

# Measurement update
unnormalized = prior * likelihood
posterior = unnormalized / unnormalized.sum()

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)

axes[0].bar(cells, prior, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_title('Prior belief (before measurement)', fontsize=10, loc='left')
axes[0].set_ylabel('P')

axes[1].bar(cells, likelihood, color='orange', edgecolor='white', linewidth=0.5)
axes[1].set_title('Likelihood p(z="landmark" | x)', fontsize=10, loc='left')
axes[1].set_ylabel('L(x)')
for lc in landmark_cells:
    axes[1].annotate(f'landmark\ncell {lc}', (lc, sensor_accuracy), ha='center', fontsize=8,
                     color='tomato', fontweight='bold')

axes[2].bar(cells, posterior, color='forestgreen', edgecolor='white', linewidth=0.5)
axes[2].set_title('Posterior belief (after measurement)', fontsize=10, loc='left')
axes[2].set_ylabel('P')
axes[2].set_xlabel('Cell index')

fig.suptitle('Measurement Update: Multiply prior by likelihood, then normalize',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Before update: entropy = {-np.sum(prior * np.log(prior + 1e-15)):.3f}')
print(f'After update:  entropy = {-np.sum(posterior * np.log(posterior + 1e-15)):.3f}')
print('Entropy decreased: the measurement made us more certain.')

## 15.4 Convergence

The real power of the Bayes filter emerges when we alternate prediction and update over many time steps. Even starting from a completely **uniform** prior (zero knowledge), the filter converges to the correct position after a few measurement cycles.

The convergence rate depends on:
- **Sensor accuracy:** better sensors mean faster convergence
- **Motion noise:** less noise means sharper predictions
- **Landmark density:** more landmarks provide more information per step
- **Ambiguity:** if the environment has repeated patterns, the filter may remain multimodal longer

Let us simulate a full run: a robot moves through the hallway, sensing at every step, and we watch the belief evolve from uniform to peaked.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_cells_conv    = 30           # hallway length             (try 20, 30, 50)
doors_conv      = [3, 10, 18, 25]  # door positions        (try different combos)
p_sense_correct = 0.75         # sensor correctness         (try 0.6, 0.75, 0.95)
motion_noise_k  = [0.1, 0.2, 0.4, 0.2, 0.1]  # motion blur kernel
n_total_steps   = 20           # total move+sense cycles    (try 10, 20, 40)
true_start      = 3            # robot's true starting cell (try 0, 3, 15)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(17)

cells = np.arange(n_cells_conv)
kernel_conv = np.array(motion_noise_k)
kernel_conv /= kernel_conv.sum()

def sense_update(bel, measurement_is_door, doors, p_correct):
    lik = np.array([
        p_correct if (i in doors) == measurement_is_door else (1 - p_correct)
        for i in range(len(bel))
    ])
    post = bel * lik
    return post / post.sum()

def motion_predict(bel, shift, kernel):
    shifted = np.roll(bel, shift)
    padded = np.pad(shifted, len(kernel), mode='wrap')
    conv = np.convolve(padded, kernel, mode='same')
    result = conv[len(kernel):-len(kernel)]
    result = np.clip(result, 0, None)
    return result / result.sum()

# Run the filter
belief = np.ones(n_cells_conv) / n_cells_conv
true_pos = true_start
history = [belief.copy()]
true_positions = [true_pos]
max_probs = [belief.max()]
entropies = [-np.sum(belief * np.log(belief + 1e-15))]

for step in range(n_total_steps):
    # Sense
    at_door = true_pos in doors_conv
    # Simulate noisy sensor
    if np.random.random() < p_sense_correct:
        sensed_door = at_door
    else:
        sensed_door = not at_door
    belief = sense_update(belief, sensed_door, doors_conv, p_sense_correct)

    # Move
    true_pos = (true_pos + 1) % n_cells_conv
    belief = motion_predict(belief, 1, kernel_conv)

    history.append(belief.copy())
    true_positions.append(true_pos)
    max_probs.append(belief.max())
    entropies.append(-np.sum(belief * np.log(belief + 1e-15)))

# Plot selected snapshots and convergence metrics
show_steps = [0, 1, 3, 7, n_total_steps]
show_steps = [s for s in show_steps if s <= n_total_steps]

fig, axes = plt.subplots(len(show_steps) + 1, 1, figsize=(12, 2.2 * (len(show_steps) + 1)))

for i, s in enumerate(show_steps):
    ax = axes[i]
    colors = ['orange' if c in doors_conv else 'steelblue' for c in cells]
    ax.bar(cells, history[s], color=colors, edgecolor='white', linewidth=0.5)
    ax.axvline(true_positions[s], color='tomato', lw=2.5, ls='--', alpha=0.8)
    ax.set_ylabel('P')
    ax.set_title(f'Step {s}: max P = {history[s].max():.3f}', fontsize=10, loc='left')
    ax.set_ylim(0, 0.35)

# Entropy over time
ax_ent = axes[-1]
ax_ent.plot(range(n_total_steps + 1), entropies, 'steelblue', lw=2, marker='o', markersize=3)
ax_ent.set_xlabel('Step')
ax_ent.set_ylabel('Entropy')
ax_ent.set_title('Entropy over time (lower = more certain)', fontsize=10, loc='left')

fig.suptitle('Convergence: from uniform prior to confident localization',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Final entropy: {entropies[-1]:.3f} (started at {entropies[0]:.3f})')
print(f'Most likely cell: {np.argmax(history[-1])}, true position: {true_positions[-1]}')

## Capstone: 2D Grid Bayes Filter

Now let us go to two dimensions. The robot lives on a $50 \times 50$ grid. There are 5 landmarks at known positions. The robot can sense the **range** to each landmark (with noise). It moves one cell at a time in a known direction (with noise).

We start with a **uniform** 2D belief (the heatmap is flat). As the robot moves and senses, watch the belief converge from a uniform fog to a bright peak at the true position.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
grid_size       = 50                             # grid dimension (50x50)   (try 30, 50)
landmarks_2d    = [(10, 10), (10, 40), (40, 10), (40, 40), (25, 25)]  # landmark positions
range_noise_std = 3.0                            # sensor noise std (cells) (try 1.0, 3.0, 6.0)
motion_noise_2d = 0.8                            # motion noise std         (try 0.3, 0.8, 2.0)
n_steps_2d      = 30                             # total steps              (try 10, 30, 50)
true_start_2d   = (5, 5)                         # starting position        (try (5,5), (25,25))
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

# Precompute range from every cell to every landmark
gy, gx = np.mgrid[0:grid_size, 0:grid_size]  # row, col
range_to_landmarks = np.zeros((grid_size, grid_size, len(landmarks_2d)))
for k, (lx, ly) in enumerate(landmarks_2d):
    range_to_landmarks[:, :, k] = np.sqrt((gx - lx)**2 + (gy - ly)**2)

def measurement_likelihood_2d(measured_ranges, noise_std):
    """Compute p(z | x) for every cell on the grid."""
    log_lik = np.zeros((grid_size, grid_size))
    for k in range(len(landmarks_2d)):
        diff = range_to_landmarks[:, :, k] - measured_ranges[k]
        log_lik -= 0.5 * (diff / noise_std) ** 2
    lik = np.exp(log_lik - log_lik.max())  # subtract max for numerical stability
    return lik

def motion_update_2d(bel, dx, dy, noise_std):
    """Shift and blur the 2D belief."""
    # Shift
    shifted = np.roll(np.roll(bel, dy, axis=0), dx, axis=1)
    # Blur with Gaussian-like kernel
    if noise_std > 0:
        ksize = max(3, int(4 * noise_std) | 1)  # odd kernel size
        ax = np.arange(ksize) - ksize // 2
        kx, ky = np.meshgrid(ax, ax)
        kernel_2d = np.exp(-0.5 * (kx**2 + ky**2) / noise_std**2)
        kernel_2d /= kernel_2d.sum()
        from scipy.signal import fftconvolve
        blurred = fftconvolve(shifted, kernel_2d, mode='same')
        blurred = np.clip(blurred, 0, None)
    else:
        blurred = shifted
    return blurred / blurred.sum()

# Simulate robot motion: move in a zig-zag pattern
true_x, true_y = true_start_2d
belief_2d = np.ones((grid_size, grid_size)) / (grid_size * grid_size)

# Generate trajectory: right, then up, then right, then down, ...
moves = []
for i in range(n_steps_2d):
    phase = i % 20
    if phase < 8:
        moves.append((1, 0))   # right
    elif phase < 12:
        moves.append((0, 1))   # up
    elif phase < 18:
        moves.append((1, 0))   # right
    else:
        moves.append((0, -1))  # down

snapshots_2d = [('Step 0 (uniform)', belief_2d.copy(), true_x, true_y)]
record_at = [1, 3, 7, 15, n_steps_2d - 1]

for step in range(n_steps_2d):
    dx, dy = moves[step]
    true_x = np.clip(true_x + dx, 0, grid_size - 1)
    true_y = np.clip(true_y + dy, 0, grid_size - 1)

    # Motion update
    belief_2d = motion_update_2d(belief_2d, dx, dy, motion_noise_2d)

    # Simulate range measurements
    true_ranges = np.array([
        np.sqrt((true_x - lx)**2 + (true_y - ly)**2)
        for lx, ly in landmarks_2d
    ])
    measured_ranges = true_ranges + np.random.normal(0, range_noise_std, len(landmarks_2d))

    # Measurement update
    lik = measurement_likelihood_2d(measured_ranges, range_noise_std)
    belief_2d = belief_2d * lik
    belief_2d /= belief_2d.sum()

    if step in record_at:
        snapshots_2d.append((f'Step {step + 1}', belief_2d.copy(), true_x, true_y))

# Plot snapshots as heatmaps
n_plots = len(snapshots_2d)
fig, axes = plt.subplots(1, n_plots, figsize=(4 * n_plots, 4))

for ax, (label, bel, tx, ty) in zip(axes, snapshots_2d):
    im = ax.imshow(bel, origin='lower', cmap='viridis', aspect='equal')
    ax.plot(tx, ty, 'r*', markersize=14, markeredgecolor='white', markeredgewidth=0.5)
    for lx, ly in landmarks_2d:
        ax.plot(lx, ly, 's', color='orange', markersize=8, markeredgecolor='white')
    ax.set_title(label, fontsize=10)
    ax.set_xlim(0, grid_size - 1)
    ax.set_ylim(0, grid_size - 1)

fig.suptitle('2D Grid Bayes Filter: convergence from uniform to peaked\n'
             '(red star = true position, orange squares = landmarks)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

peak = np.unravel_index(np.argmax(belief_2d), belief_2d.shape)
print(f'True position:  ({true_x}, {true_y})')
print(f'Peak of belief: ({peak[1]}, {peak[0]})')
print(f'Max probability: {belief_2d.max():.6f}')

## Exercises

**Exercise 15.1: Build the 1D hallway filter from scratch.**

Create a hallway with 25 cells and doors at cells [2, 8, 14, 21]. Implement `predict(belief, move, kernel)` and `update(belief, measurement, door_positions, p_correct)` yourself. Start from a uniform prior. The robot moves right by 1 cell per step and senses at every step. The true robot starts at cell 2. Run for 15 steps and plot the final belief.

Hint: for `predict`, use `np.roll` for the shift and `np.convolve` (with `mode='same'`) for the blur.

In [ ]:
# Your code here


**Exercise 15.2: Effect of sensor quality.**

Using the hallway from Exercise 15.1, run the filter with three different sensor accuracies: `p_correct = 0.6`, `0.8`, and `0.99`. Plot the entropy over time for all three on the same graph. Which converges fastest? At what accuracy does the filter become practically useless?

In [ ]:
# Your code here


**Exercise 15.3: Symmetric environments.**

Create a hallway with 20 cells and doors at cells [5, 15] (symmetric). Start the robot at cell 5. Run the filter for 30 steps. Does the belief ever fully resolve the ambiguity between the two door positions? Now add a third door at cell 12 (breaking the symmetry). Run again. How does this change convergence?

In [ ]:
# Your code here


**Exercise 15.4: Motion noise sensitivity.**

Use a 40 cell hallway with doors at [5, 15, 25, 35]. Compare three motion kernels:
- Precise: `[0.05, 0.9, 0.05]`
- Moderate: `[0.1, 0.2, 0.4, 0.2, 0.1]`
- Sloppy: `[0.15, 0.15, 0.15, 0.1, 0.15, 0.15, 0.15]`

Run 25 steps for each. Plot the maximum probability at each step for all three on one graph. What happens to the filter when motion noise is very large?

In [ ]:
# Your code here


**Exercise 15.5 (Challenge): Circular hallway with kidnapping.**

Create a 30 cell circular hallway with doors at [3, 12, 22]. The robot starts at cell 3 and moves right. After 15 steps, "kidnap" the robot by teleporting it to cell 22 (but do not reset the belief). Run for 15 more steps. Plot the belief at steps 14 (just before kidnap), 16 (just after), 20, and 30. How many steps does the filter need to recover? What happens if you inject a small amount of uniform noise at every step (add $\epsilon / n$ to every cell and renormalize)? Implement this "recovery noise" and show it helps.

In [ ]:
# Your code here
